In [ ]:
suppressMessages(suppressWarnings(library(clusterProfiler)))
suppressMessages(suppressWarnings(library(readxl)))
suppressMessages(suppressWarnings(library(tidyverse)))
suppressMessages(suppressWarnings(library(DESeq2)))
suppressMessages(suppressWarnings(library(WGCNA)))
suppressMessages(suppressWarnings(library(ggpubr)))
suppressMessages(suppressWarnings(library(reshape2)))
suppressMessages(suppressWarnings(library(cowplot)))
suppressMessages(suppressWarnings(library(limma)))
suppressMessages(suppressWarnings(library(edgeR)))
suppressMessages(suppressWarnings(library(Seurat)))
suppressMessages(suppressWarnings(library(utils)))
suppressMessages(suppressWarnings(library(openxlsx)))
options(repr.plot.width = 15, repr.plot.height = 15)

meta <- read_excel("../metadata_X1_SCOP-2024-0277.xlsx", sheet = 1)[-c(5, 69),] %>%
  rename_with(~ "Anatomy", .cols = 10) %>%
  rename_with(~ "Sample_ID", .cols = 4) %>%
  mutate(Identifier = paste0(Condition1, "_", Anatomy, ".", `Wet lab ID`))

data <- read.table("../salmon.merged.gene_counts.tsv", sep = "\t", header = TRUE) %>%
  select(-gene_id) %>%
  filter(!duplicated(gene_name)) %>%
  column_to_rownames("gene_name") %>%
  round()

colnames(data) <- meta$Identifier

# Prepare data

## Hypothalamus

In [ ]:
hypo <- data[, grep("Hypothalamus", colnames(data))]
hypo_meta <- meta %>% filter(grepl("Hypothalamus", Identifier))
hypo <- hypo[, hypo_meta$Identifier]
hypo_colData <- DataFrame(condition = factor(hypo_meta$Condition1, levels = c("Vehicle", "Cagrilinitide", "Retatrutide", "Combo")))

In [ ]:
hypo_dge <- DGEList(counts = hypo, group = hypo_colData$condition)
hypo_dge <- hypo_dge[filterByExpr(hypo_dge), , keep.lib.sizes = FALSE]
hypo_dge <- calcNormFactors(hypo_dge, method = "TMM")

In [ ]:
design_hypo <- model.matrix(~0 + group, data = data.frame(group = hypo_dge$samples$group))
colnames(design_hypo) <- levels(hypo_dge$samples$group)

In [ ]:
v_hypo <- voomWithQualityWeights(hypo_dge, design = design_hypo, plot = FALSE)

hypo_norm_counts <- v_hypo$E

## DVC

In [ ]:
DVC <- data[, grep("DVC", colnames(data))]
DVC_meta <- meta %>% filter(grepl("DVC", Identifier))
DVC <- DVC[, DVC_meta$Identifier]
DVC_colData <- DataFrame(condition = factor(DVC_meta$Condition1, levels = c("Vehicle", "Cagrilinitide", "Retatrutide", "Combo")))

In [ ]:
DVC_dge <- DGEList(counts = DVC, group = DVC_colData$condition)
DVC_dge <- DVC_dge[filterByExpr(DVC_dge), , keep.lib.sizes = FALSE]
DVC_dge <- calcNormFactors(DVC_dge, method = "TMM")

In [ ]:
design_DVC <- model.matrix(~0 + group, data = data.frame(group = DVC_dge$samples$group))
colnames(design_DVC) <- levels(DVC_dge$samples$group)

In [ ]:
v_DVC <- voomWithQualityWeights(DVC_dge, design = design_DVC, plot = FALSE)

DVC_norm_counts <- v_DVC$E

# WGCNA

## Functions

In [ ]:
run_bulk_WGCNA <- function(data, softpower) {

  input_mat <- t(data)

  # ----------------- Adjacency / TOM / clustering -----------------
  adjacency <- adjacency(
    input_mat,
    power  = softpower,
    corFnc = "bicor",
    type   = "signed"
  )

  TOM <- TOMsimilarity(adjacency, TOMType = "signed")
  TOM_dissimilarity <- 1 - TOM

  gene_tree <- hclust(as.dist(TOM_dissimilarity), method = "average")

  dynamic_mods <- cutreeDynamic(
    dendro             = gene_tree,
    distM              = TOM_dissimilarity,
    deepSplit          = 3,
    pamRespectsDendro  = FALSE,
    minClusterSize     = 30,
    pamStage           = FALSE
  )
  dynamic_colors <- labels2colors(dynamic_mods)

  # ----------------- Eigengenes and module merging -----------------
  MEs0 <- moduleEigengenes(input_mat, colors = dynamic_colors)$eigengenes
  ME_dissimilarity <- 1 - cor(MEs0)
  ME_tree <- hclust(as.dist(ME_dissimilarity), method = "average")

  merge <- mergeCloseModules(
    exprData = input_mat,
    colors   = dynamic_colors,
    cutHeight = 0.25,
    verbose   = 3
  )

  merged_colors <- merge$colors
  names(merged_colors) <- colnames(input_mat)

  plotDendroAndColors(
    dendro       = gene_tree,
    colors       = merged_colors,
    groupLabels  = "Gene dendrogram and merged module colors",
    dendroLabels = FALSE,
    hang         = 0.03,
    addGuide     = TRUE,
    guideHang    = 0.05
  )

  MEs <- merge$newMEs     # samples x (merged modules)
  rownames(MEs) <- rownames(input_mat)

  # ----------------- Gene → module assignment (M1, M2, …) ---------
  module_index <- match(paste0("ME", merged_colors), colnames(MEs))

  gene_assignment <- data.frame(
    module = ifelse(is.na(module_index), NA, paste0("M", module_index)),
    stringsAsFactors = FALSE
  )
  rownames(gene_assignment) <- colnames(input_mat)

  gene_assignment$module <- factor(
    gene_assignment$module,
    levels = paste0("M", seq_len(ncol(MEs)))
  )

  # ----------------- Network list by module ------------------------
  network <- list()
  for (i in levels(gene_assignment$module)) {
    genes <- rownames(gene_assignment)[gene_assignment$module == i]
    network[[i]] <- genes
  }

  colnames(MEs) <- names(network)

  # ----------------- Return -----------------
  return(list(MEs, network, adjacency))
}

In [ ]:
run_linear_model <- function(MEs, model_design, model_contrast) {

  # Run linear model
  significance <- data.frame(matrix(0, nrow = 2, ncol = ncol(MEs)))
  colnames(significance) <- colnames(MEs)
  rownames(significance) <- c("pval", "adjusted_pval")

  for (i in 1:ncol(MEs)) {
      
    ME <- MEs[, i]

    # Run linear model
    fit <- lmFit(ME, model_design)
    fit2 <- contrasts.fit(fit, model_contrast)
    fit2 <- eBayes(fit2)

    # Store results
    significance[1, i] <- topTable(fit2)$P.Value
  }

  significance[2, ] <- p.adjust(unlist(significance[1, ]), method = "BH")

  return(significance)
}

## Hypothalamus

In [ ]:
# Find power
powers <- c(c(1:10), seq(from = 12, to = 20, by = 2))

input_mat <- t(hypo_norm_counts)
sft <- pickSoftThreshold(input_mat, powerVector = powers, verbose = 5, 
                       networkType = "signed", corFnc = "bicor")

plot(sft$fitIndices[, 1],
     -sign(sft$fitIndices[, 3]) * sft$fitIndices[, 2],
     xlab = "Soft Threshold (power)",
     ylab = "Scale Free Topology Model Fit, signed R^2",
     type = "n")
text(sft$fitIndices[, 1],
     -sign(sft$fitIndices[, 3]) * sft$fitIndices[, 2],
     labels = powers, cex = 0.9, col = "red")
abline(h = 0.90, col = "red")

In [ ]:
wgcna_hypo <- run_bulk_WGCNA(hypo_norm_counts, 10)
MEs_hypo <- wgcna_hypo[[1]]
network_hypo <- wgcna_hypo[[2]]
adjacency_hypo <- wgcna_hypo[[3]]

In [ ]:
# Examine number of genes in each module
lapply(network_hypo, function(x) { length(x) }) 

In [ ]:
MEs_hypo <- MEs_hypo %>% select(-M18)
network_hypo <- network_hypo[!names(network_hypo) %in% "M18"]

In [ ]:
kME_hypo <- signedKME(
  datExpr = t(hypo_norm_counts),
  datME   = MEs_hypo,
  corFnc  = "bicor",
  outputColumnName = "kME"
)

colnames(kME_hypo) <- paste0("kMEM", seq(1:17))

### Linear model

In [ ]:
# Look for significant modules
lm_hypo <- data.frame(rownames = rownames(MEs_hypo)) %>%
  mutate(group = factor(sub("\\.\\d+$", "", rownames)),
         treatment = factor(case_when(
           grepl("Vehicle", group) ~ "Vehicle",
           grepl("Cagri", group) ~ "Cagrilinitide",
           grepl("Reta", group) ~ "Retatrutide",
           grepl("Combo", group) ~ "Combo"),
           levels = c("Vehicle", "Cagrilinitide", "Retatrutide", "Combo")
         ),
         weight_change = meta$`deltaWeight (%)`[match(rownames(MEs_hypo), meta$Identifier)]
         ) %>%
  column_to_rownames(var = "rownames")

design_hypo <- model.matrix(~ 0 + treatment, data = lm_hypo)
colnames(design_hypo) <- gsub("treatment", "", colnames(design_hypo))

In [ ]:
hypo_vehicle_cagri <- run_linear_model(MEs = MEs_hypo,
                   model_design = design_hypo,
                   model_contrast = makeContrasts(contrast = Vehicle - Cagrilinitide, levels = design_hypo))
hypo_vehicle_reta <- run_linear_model(MEs = MEs_hypo,
                   model_design = design_hypo,
                   model_contrast = makeContrasts(contrast = Vehicle - Retatrutide, levels = design_hypo))
hypo_vehicle_combo <- run_linear_model(MEs = MEs_hypo,
                   model_design = design_hypo,
                   model_contrast = makeContrasts(contrast = Vehicle - Combo, levels = design_hypo))

In [ ]:
moi_hypo_cagri <- MEs_hypo[, hypo_vehicle_cagri[2, ] < 0.05, drop = FALSE]               
moi_hypo_reta <- MEs_hypo[, hypo_vehicle_reta[2, ] < 0.05, drop = FALSE]              
moi_hypo_combo <- MEs_hypo[, hypo_vehicle_combo[2, ] < 0.05, drop = FALSE]

# M18 is the grey/leftover module and should not be considered
paste0("Cagri vehicle: ", colnames(moi_hypo_cagri))
paste0("Reta vehicle: ", colnames(moi_hypo_reta))
paste0("Combo vehicle: ", colnames(moi_hypo_combo))

## DVC

In [ ]:
# Find power
powers <- c(c(1:10), seq(from = 12, to = 20, by = 2))

input_mat <- t(DVC_norm_counts)
sft <- pickSoftThreshold(input_mat, powerVector = powers, verbose = 5, 
                       networkType = "signed", corFnc = "bicor")

plot(sft$fitIndices[, 1],
     -sign(sft$fitIndices[, 3]) * sft$fitIndices[, 2],
     xlab = "Soft Threshold (power)",
     ylab = "Scale Free Topology Model Fit, signed R^2",
     type = "n")
text(sft$fitIndices[, 1],
     -sign(sft$fitIndices[, 3]) * sft$fitIndices[, 2],
     labels = powers, cex = 0.9, col = "red")
abline(h = 0.90, col = "red")

In [ ]:
wgcna_DVC <- run_bulk_WGCNA(DVC_norm_counts, 18)
MEs_DVC <- wgcna_DVC[[1]]
network_DVC <- wgcna_DVC[[2]]
adjacency_DVC <- wgcna_DVC[[3]]

In [ ]:
lapply(network_DVC, function(x) { length(x) }) 

In [ ]:
MEs_DVC <- MEs_DVC %>% select(-M36)
network_DVC <- network_DVC[!names(network_DVC) %in% "M36"]

In [ ]:
kME_DVC <- signedKME(
  datExpr = t(DVC_norm_counts),
  datME   = MEs_DVC,
  corFnc  = "bicor",
  outputColumnName = "kME"
)

colnames(kME_DVC) <- paste0("kMEM", seq(1:35))

### Linear model

In [ ]:
# DVC
lm_DVC <- data.frame(rownames = rownames(MEs_DVC)) %>%
  mutate(group = factor(sub("\\.\\d+$", "", rownames)),
         treatment = factor(case_when(
           grepl("Vehicle", group) ~ "Vehicle",
           grepl("Cagri", group) ~ "Cagrilinitide",
           grepl("Reta", group) ~ "Retatrutide",
           grepl("Combo", group) ~ "Combo")),
         weight_change = meta$`deltaWeight (%)`[match(rownames(MEs_DVC), meta$Identifier)]
         ) %>%
  column_to_rownames(var = "rownames")

design_DVC <- model.matrix(~ 0 + treatment, data = lm_DVC)
colnames(design_DVC) <- gsub("treatment", "", colnames(design_DVC))

In [ ]:
DVC_vehicle_cagri <- run_linear_model(MEs = MEs_DVC,
                   model_design = design_DVC,
                   model_contrast = makeContrasts(contrast = Vehicle - Cagrilinitide, levels = design_DVC))
DVC_vehicle_reta <- run_linear_model(MEs = MEs_DVC,
                   model_design = design_DVC,
                   model_contrast = makeContrasts(contrast = Vehicle - Retatrutide, levels = design_DVC))
DVC_vehicle_combo <- run_linear_model(MEs = MEs_DVC,
                   model_design = design_DVC,
                   model_contrast = makeContrasts(contrast = Vehicle - Combo, levels = design_DVC))

In [ ]:
moi_DVC_cagri <- MEs_DVC[, DVC_vehicle_cagri[2, ] < 0.05, drop = FALSE]               
moi_DVC_reta <- MEs_DVC[, DVC_vehicle_reta[2, ] < 0.05, drop = FALSE]              
moi_DVC_combo <- MEs_DVC[, DVC_vehicle_combo[2, ] < 0.05, drop = FALSE]

# M36 is the grey/leftover module and should not be considered
paste0("Cagri vehicle: ", colnames(moi_DVC_cagri))
paste0("Reta vehicle: ", colnames(moi_DVC_reta))
paste0("Combo vehicle: ", colnames(moi_DVC_combo))

## Where are the important receptors located?

### DVC

In [ ]:
cellex_rat <- read.table("../neurons_rat_2025.esmu.csv", header = TRUE, sep = ",", row.names = 1)
colnames(cellex_rat) <- gsub("_(\\d+)_", "\\1.", colnames(cellex_rat))

In [ ]:
cellex_rat <- cellex_rat %>% select(c("Chat0.0", "GABA0.0", "GABA0.1", "GABA0.2", # Spatially confirmed DVC
                         "GABA2.0", "GABA2.1", "GABA2.2", "GABA6.1",
                         "GABA6.2", "Glu0.2", "Glu2.1", "Glu2.2", 
                         "Glu2.3", "Glu2.4", "Glu3.0", "Glu3.1", 
                         "Glu3.2", "Glu3.3", "Glu3.4", "Glu4.0", 
                         "Glu4.1", "Glu4.2", "Glu4.3", "Glu5.0", 
                         "Glu5.1", "Glu5.2", "Glu5.3", "Glu6.0",
                         "Glu6.1", "Glu6.2", "Glu8.0", "Glu8.1", 
                         "Glu8.2", "Glu8.3"))

In [ ]:
genes <- c("Glp1r","Gcgr","Gipr","Calcr","Ramp1","Ramp3")
coi <- colnames(cellex_rat)[colSums(cellex_rat[genes, , drop = FALSE] > 0.8) > 0]  # At least one of the listed genes with an ESMu > 0.8
# cellex_fdr_coi <- cellex_fdr %>% filter(cell.type %in% coi)
# area_coi <- area %>% filter(cell.type %in% coi)


hmap_data <- cellex_rat[genes, coi]

In [ ]:
ComplexHeatmap::Heatmap(
  (as.matrix(hmap_data[, gtools::mixedsort(colnames(hmap_data)), drop = FALSE]) >= 0.8) * 1,,
  name = "> 0.8",
  col = c("#f0f0f0", "#d7301f"),                 # 0 = grey, 1 = red
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  show_row_names = TRUE,
  show_column_names = TRUE,
  row_names_side = "left",
  heatmap_legend_param = list(at = c(0, 1), labels = c("< 0.8", "> 0.8")),
  rect_gp = grid::gpar(col = "grey80", lwd = 0.6), # gridlines
  use_raster = FALSE                                # ensure borders are drawn
)

### Hypothalamus

In [ ]:
cellex_mouse <- read.table("../FGF1_CELLEX.esmu.csv", header = TRUE, sep = ",", row.names = 1)  # Using mouse as surrogate

In [ ]:
genes <- c("Glp1r", "Gipr","Calcr","Ramp1","Ramp3")
coi <- colnames(cellex_mouse)[colSums(cellex_mouse[genes, , drop = FALSE] > 0.8) > 0]  # At least one of the listed genes with an ESMu > 0.8
# cellex_fdr_coi <- cellex_fdr %>% filter(cell.type %in% coi)
# area_coi <- area %>% filter(cell.type %in% coi)


hmap_data <- cellex_mouse[genes, coi]

In [ ]:
ComplexHeatmap::Heatmap(
  (as.matrix(hmap_data[, gtools::mixedsort(colnames(hmap_data)), drop = FALSE]) >= 0.8) * 1,,
  name = "> 0.8",
  col = c("#f0f0f0", "#d7301f"),                 # 0 = grey, 1 = red
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  show_row_names = TRUE,
  show_column_names = TRUE,
  row_names_side = "left",
  heatmap_legend_param = list(at = c(0, 1), labels = c("< 0.8", "> 0.8")),
  rect_gp = grid::gpar(col = "grey80", lwd = 0.6), # gridlines
  use_raster = FALSE                                # ensure borders are drawn
)

## Module cell type enrichment

### DVC

In [ ]:
cellex_rat <- read.table("../neurons_rat_2025.esmu.csv", header = TRUE, sep = ",", row.names = 1)
colnames(cellex_rat) <- gsub("_(\\d+)_", "\\1.", colnames(cellex_rat))

In [ ]:
cellex_rat <- cellex_rat %>% select(c("Chat0.0", "GABA0.0", "GABA0.1", "GABA0.2", # Spatially confirmed DVC
                         "GABA2.0", "GABA2.1", "GABA2.2", "GABA6.1",
                         "GABA6.2", "Glu0.2", "Glu2.1", "Glu2.2", 
                         "Glu2.3", "Glu2.4", "Glu3.0", "Glu3.1", 
                         "Glu3.2", "Glu3.3", "Glu3.4", "Glu4.0", 
                         "Glu4.1", "Glu4.2", "Glu4.3", "Glu5.0", 
                         "Glu5.1", "Glu5.2", "Glu5.3", "Glu6.0",
                         "Glu6.1", "Glu6.2", "Glu8.0", "Glu8.1", 
                         "Glu8.2", "Glu8.3"))
genes <- c("Glp1r","Gcgr","Gipr","Calcr","Ramp1","Ramp3")
coi <- colnames(cellex_rat)[colSums(cellex_rat[genes, , drop = FALSE] > 0.8) > 0]  # At least one of the listed genes with an ESMu > 0.8

In [ ]:
moi_DVC <- unique(c(colnames(moi_DVC_cagri), colnames(moi_DVC_reta),colnames(moi_DVC_combo)))

In [ ]:
cellex_pval <- matrix(NA, length(moi_DVC), ncol(cellex_rat)) 
rownames(cellex_pval) <- moi_DVC
colnames(cellex_pval) <- colnames(cellex_rat)

In [ ]:
for (i in moi_DVC) {
  for (j in 1:ncol(cellex_rat)) {
    
    all_genes <- intersect(rownames(cellex_rat), rownames(DVC_dge))
    module_genes <- all_genes[which(all_genes %in% network_DVC[[i]])] 
    celltype_genes <- rownames(cellex_rat)[which(cellex_rat[, j] > 0)]
    
    overlap <- intersect(module_genes, celltype_genes)
    a <- length(overlap)
    b <- length(celltype_genes) - length(overlap)
    c <- length(module_genes) - length(overlap)
    d <- length(all_genes) - a - b - c
      
    con_table <- matrix(c(a, b, c, d), nrow = 2)
    
    cellex_pval[i, j] <- fisher.test(con_table, alternative = "greater")[["p.value"]]
  } 
}

cellex_fdr <- p.adjust(cellex_pval, method = "BH")
dim(cellex_fdr) <- dim(cellex_pval)
dimnames(cellex_fdr) <- dimnames(cellex_pval)
cellex_fdr <- melt(cellex_fdr)
colnames(cellex_fdr) <- c("module", "cell.type", "p.adj")
cellex_fdr$size <- -log10(cellex_fdr$p.adj)
cellex_fdr$size[which(cellex_fdr$size < -log10(0.01))] <- NA
cellex_fdr$module <- factor(cellex_fdr$module, 
                            levels = rev(moi_DVC))
cellex_fdr_coi <- cellex_fdr %>% filter(cell.type %in% coi) %>% drop_na()

In [ ]:
is_AP <- c("GABA0.0", "GABA0.2", "Glu4.2", "Glu5.0", "Glu8.1")

is_NTS <- c("GABA0.0", "GABA0.1", "GABA2.0", "GABA2.1", "GABA2.2", "GABA6.1", "GABA6.2",
            "Glu0.2", "Glu2.1", "Glu2.2", "Glu2.3", "Glu2.4", "Glu3.0", "Glu3.1", "Glu3.2",
            "Glu3.3", "Glu3.4", "Glu4.0", "Glu4.1", "Glu4.2", "Glu4.3", "Glu5.1", "Glu5.2",
            "Glu5.3", "Glu6.0", "Glu6.1", "Glu6.2", "Glu8.0", "Glu8.1", "Glu8.2", "Glu8.3")

is_DMV <- c("Chat0.0", "GABA0.1", "Glu2.4", "Glu3.0", "Glu8.3")

areas <- list(DMV = is_DMV, NTS = is_NTS, AP = is_AP)
area <- enframe(areas, name = "area", value = "cell.type") %>%
  unnest(cell.type) %>%
  mutate(size = 1L) %>%
  complete(cell.type = colnames(cellex_rat), area = names(areas)) %>%
  mutate(area = factor(area, levels = c("DMV", "NTS", "AP")))
area_coi <- area %>% filter(cell.type %in% coi) %>% filter(cell.type %in% unique(cellex_fdr_coi$cell.type))

In [ ]:
cellex_plot <- ggplot(cellex_fdr_coi, aes(x = cell.type, y = module, color = module)) +
  geom_tile(size = 1, color = "white", fill="grey99") +
  geom_point(aes(size = size)) + 
  scale_size(name = expression(paste(bold(-log[10]),bold("("),
                                     bolditalic("P"),bold(")"))), 
             breaks = c(5, 15, 25),
             range = c(0, 5), limits = c(0, 32)) + 
  theme_pubr(legend = "top") + xlab(NULL) + ylab(NULL) + 
  theme(axis.text.x = element_blank(), 
        axis.text.y = element_text(size=10, face="bold"),
        legend.title = element_text(size=10, face="bold"), 
        legend.text = element_text(size=10, face="bold"),
        legend.key.width = unit(-0.05, "cm"),
        axis.line = element_line(colour = "black", size = 0.4),
        axis.ticks.x = element_blank(),
        # margin: top, right, bottom, and left
        plot.margin = unit(c(0, 0, 0, 0.5), "cm")) +
  scale_color_manual(values= c("#8A843E", "#73BCC9", "#CC8B93", "#025666",
                               "#8A843E", "#73BCC9", "#CC8B93", "#025666",
                               "#8A843E", "#73BCC9"), guide = F)

area_plot <- ggplot(area_coi, aes(x = cell.type, y = area)) +
  geom_tile(size = 1, color = "white", fill="grey99") +
  geom_point(aes(size = size), shape = 15, color = "grey60") + 
  scale_size(range = c(0,2)) +
  scale_color_manual(values = "black") +
  theme_pubr() + xlab(NULL) + ylab(NULL) + 
  theme(axis.text.x = element_text(angle=45, hjust=1, size=10, face="bold"), 
        legend.position = "none",
        axis.line = element_line(colour = "black", size = 0.4),
        axis.text = element_text(size=6, face="bold"),
        # margin: top, right, bottom, and left
        plot.margin = unit(c(0, 0, 0, 0.5), "cm"))

p <- plot_grid(
  cellex_plot + theme(legend.position = "none"),
  area_plot,
  ncol = 1, align = "v", axis = "rl",
  rel_heights = c(1, 0.25)   # ↓ shrink area plot (try 0.2–0.4)
)

p

In [ ]:
dvc_celltype_export <- cellex_fdr %>% select(module, cell.type, p.adj)
write.xlsx(dvc_celltype_export, "DVC_celltype_enrichment.xlsx")

### Hypothalamus

In [ ]:
moi_hypo <- unique(c(colnames(moi_hypo_cagri), colnames(moi_hypo_reta),colnames(moi_hypo_combo)))

In [ ]:
cellex_mouse <- read.table("../FGF1_CELLEX.esmu.csv", header = TRUE, sep = ",", row.names = 1)
cellex_mouse <- cellex_mouse %>% select(-unassigned.2.)

In [ ]:
genes <- c("Glp1r", "Gipr","Calcr","Ramp1","Ramp3")
coi <- colnames(cellex_mouse)[colSums(cellex_mouse[genes, , drop = FALSE] > 0.8) > 0]  # At least one of the listed genes with an ESMu > 0.8

In [ ]:
mouse_to_rat <- read.csv("../mouse_to_rat.txt", sep = "\t") # Biomart mapping file
colnames(mouse_to_rat) <- c("mouse", "rat")

In [ ]:
cellex_mouse_rownames <- as.data.frame(rownames(cellex_mouse))
colnames(cellex_mouse_rownames) <- "mouse"
cellex_mouse_rownames <- cellex_mouse_rownames %>% left_join(mouse_to_rat, by = "mouse")
cellex_mouse_rownames <- cellex_mouse_rownames %>% filter(!is.na(rat) & rat != "")
cellex_mouse_rownames <- cellex_mouse_rownames %>% distinct(rat, .keep_all = TRUE)
cellex_mouse_rownames <- cellex_mouse_rownames %>% distinct(mouse, .keep_all = TRUE)

In [ ]:
cellex_mouse <- cellex_mouse[cellex_mouse_rownames$mouse,]
rownames(cellex_mouse) <- cellex_mouse_rownames$rat

In [ ]:
cellex_pval <- matrix(NA, length(moi_hypo), ncol(cellex_mouse)) 
rownames(cellex_pval) <- moi_hypo
colnames(cellex_pval) <- colnames(cellex_mouse)

In [ ]:
for (i in moi_hypo) {
  for (j in 1:ncol(cellex_mouse)) {
    
    all_genes <- intersect(rownames(cellex_mouse), rownames(hypo_dge))
    module_genes <- all_genes[which(all_genes %in% network_hypo[[i]])] 
    celltype_genes <- rownames(cellex_mouse)[which(cellex_mouse[, j] > 0)]
    
    overlap <- intersect(module_genes, celltype_genes)
    a <- length(overlap)
    b <- length(celltype_genes) - length(overlap)
    c <- length(module_genes) - length(overlap)
    d <- length(all_genes) - a - b - c
      
    con_table <- matrix(c(a, b, c, d), nrow = 2)
    
    cellex_pval[i, j] <- fisher.test(con_table, alternative = "greater")[["p.value"]]
  } 
}

cellex_fdr <- p.adjust(cellex_pval, method = "BH")
dim(cellex_fdr) <- dim(cellex_pval)
dimnames(cellex_fdr) <- dimnames(cellex_pval)
cellex_fdr <- melt(cellex_fdr)
colnames(cellex_fdr) <- c("module", "cell.type", "p.adj")
cellex_fdr$size <- -log10(cellex_fdr$p.adj)
cellex_fdr$size[which(cellex_fdr$size < -log10(0.01))] <- NA
cellex_fdr$module <- factor(cellex_fdr$module, 
                            levels = rev(moi_hypo))
cellex_fdr_coi <- cellex_fdr %>% filter(cell.type %in% coi) %>% drop_na()

In [ ]:
cellex_plot <- ggplot(cellex_fdr_coi, aes(x = cell.type, y = module, color = module)) +
  geom_tile(size = 1, color = "white", fill="grey99") +
  geom_point(aes(size = size)) + 
  scale_size(name = expression(paste(bold(-log[10]),bold("("),
                                     bolditalic("P"),bold(")"))), 
             breaks = c(5, 15, 25),
             range = c(0, 5), limits = c(0, 32)) + 
  theme_pubr(legend = "top") +
  xlab(NULL) + ylab(NULL) + 
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),  # RESTORED LABELS
    axis.text.y = element_text(size = 10, face = "bold"),
    legend.title = element_text(size = 10, face = "bold"), 
    legend.text = element_text(size = 10, face = "bold"),
    legend.key.width = unit(-0.05, "cm"),
    axis.line = element_line(colour = "black", size = 0.4),
    axis.ticks.x = element_line()
  ) +
  scale_color_manual(values = c("#8A843E", "#73BCC9", "#CC8B93", "#025666",
                                "#8A843E", "#73BCC9", "#CC8B93", "#025666",
                                "#8A843E", "#73BCC9"),
                     guide = FALSE)

cellex_plot

In [ ]:
hypo_celltype_export <- cellex_fdr %>% select(module, cell.type, p.adj)
write.xlsx(hypo_celltype_export, "Hypothalamus_celltype_enrichment.xlsx")

## BMI GWAS gene enrichment

In [ ]:
bmi_gwas <- read.table("../BMI_Yengo2018.genes.out", header = TRUE)  # MAGMA output
entrez_map <- read.csv("../entrez_to_ensembl_human.txt", sep = "\t")  # Biomart mapping file
colnames(entrez_map) <- c("ensembl", "entrez")
rat_map <- read.csv("../human-ens_to_rat.txt", sep = "\t")[1:2]  # Biomart mapping file
colnames(rat_map) <- c("human_ens", "rat_gene")

rat_map_unique <- rat_map %>%
  group_by(human_ens) %>%
  filter(n() == 1) %>%
  ungroup()

bmi_gwas <- bmi_gwas %>%
  inner_join(entrez_map, by = c("GENE" = "entrez")) %>%  # Mapping entrez ID (the number in GENE col) to ensembl
  inner_join(rat_map_unique, by = c("ensembl" = "human_ens"))  # Mapping human ensembl to rat gene name

In [ ]:
bmi_gwas$adj <- p.adjust(bmi_gwas$P, method = "bonferroni")
bmi_gwas_genes <- bmi_gwas %>% filter(adj < 0.05 & rat_gene != "") %>% distinct(rat_gene) %>% pull(rat_gene)

In [ ]:
rare_variant_bmi <- c(
  "ENSG00000100578","ENSG00000138688","ENSG00000087085","ENSG00000116127","ENSG00000107282",
  "ENSG00000105186","ENSG00000151572","ENSG00000113966","ENSG00000174483","ENSG00000179941",
  "ENSG00000181004","ENSG00000125124","ENSG00000140463","ENSG00000163093","ENSG00000138686",
  "ENSG00000122507","ENSG00000176697","ENSG00000164061","ENSG00000004948","ENSG00000174007",
  "ENSG00000198707","ENSG00000142002","ENSG00000197586","ENSG00000010310","ENSG00000087460",
  "ENSG00000147257","ENSG00000173250","ENSG00000119737","ENSG00000130787","ENSG00000171435",
  "ENSG00000174697","ENSG00000116678","ENSG00000163818","ENSG00000166603","ENSG00000125863",
  "ENSG00000011143","ENSG00000148053","ENSG00000046651","ENSG00000175426","ENSG00000152270",
  "ENSG00000156531","ENSG00000115138","ENSG00000181929","ENSG00000108946","ENSG00000080371",
  "ENSG00000112210","ENSG00000108557","ENSG00000079337","ENSG00000169855","ENSG00000054282",
  "ENSG00000178188","ENSG00000112246","ENSG00000113140","ENSG00000135111","ENSG00000119401",
  "ENSG00000165533","ENSG00000024048","ENSG00000152332","ENSG00000132549","ENSG00000143951",
  "ENSG00000169884","ENSG00000160685","ENSG00000140836","ENSG00000105278","ENSG00000175787",
  "ENSG00000196277","ENSG00000138688","ENSG00000138688","ENSG00000101191","ENSG00000144724",
  "ENSG00000124140","ENSG00000137776"
)

rare_variant_bmi_genes <- rat_map_unique[which(rat_map_unique$human_ens %in% rare_variant_bmi),]$rat_gene

### DVC

In [ ]:
options(scipen = 999)

wgcna_genes   <- unique(unlist(network_DVC))
gwas_all      <- unique(bmi_gwas$rat_gene)  # all genes with GWAS p-values, not just sig
bg_dvc            <- intersect(wgcna_genes, gwas_all)
gwas_dvc      <- intersect(bmi_gwas_genes, bg_dvc)  # significant GWAS genes in bg

enrich_module <- function(m) {
  mod_genes <- intersect(network_DVC[[m]], bg_dvc)

  a <- length(intersect(mod_genes, gwas_dvc))                  # in module & GWAS
  b <- length(setdiff(mod_genes, gwas_dvc))                    # in module & not GWAS
  c <- length(setdiff(gwas_dvc, mod_genes))                    # in GWAS & not module
  d <- length(setdiff(bg_dvc, c(mod_genes, gwas_dvc)))             # neither

  ft  <- fisher.test(matrix(c(a, b, c, d), nrow = 2),
                     alternative = "greater")

  data.frame(
    module      = m,
    overlap     = a,
    module_size = length(mod_genes),
    p           = ft$p.value,
    stringsAsFactors = FALSE
  )
}

enrichment_results_DVC <- do.call(rbind, lapply(names(network_DVC), enrich_module)) %>%
  mutate(p_fdr  = p.adjust(p, method = "fdr"))

enrichment_results_DVC[enrichment_results_DVC$p_fdr < 0.05,]

### Hypothalamus

In [ ]:
options(scipen = 999)

wgcna_genes   <- unique(unlist(network_hypo))
gwas_all      <- unique(bmi_gwas$rat_gene)  # all genes with GWAS p-values, not just sig
bg_hypo            <- intersect(wgcna_genes, gwas_all)
gwas_hypo      <- intersect(bmi_gwas_genes, bg_hypo)  # significant GWAS genes in bg

enrich_module <- function(m) {
  mod_genes <- intersect(network_hypo[[m]], bg_hypo)

  a <- length(intersect(mod_genes, gwas_hypo))                  # in module & GWAS
  b <- length(setdiff(mod_genes, gwas_hypo))                    # in module & not GWAS
  c <- length(setdiff(gwas_hypo, mod_genes))                    # in GWAS & not module
  d <- length(setdiff(bg_hypo, c(mod_genes, gwas_hypo)))             # neither

  ft  <- fisher.test(matrix(c(a, b, c, d), nrow = 2),
                     alternative = "greater")

  data.frame(
    module      = m,
    overlap     = a,
    module_size = length(mod_genes),
    p           = ft$p.value,
    stringsAsFactors = FALSE
  )
}

enrichment_results_hypo <- do.call(rbind, lapply(names(network_hypo), enrich_module)) %>%
  mutate(p_fdr  = p.adjust(p, method = "fdr"))

enrichment_results_hypo[enrichment_results_hypo$p_fdr < 0.05,]

## GO

In [ ]:
hypo_ego_M4 <- enrichGO(
  gene = intersect(intersect(network_hypo[["M4"]], bg_hypo), gwas_hypo),
  universe = rownames(hypo_norm_counts),
  keyType = "SYMBOL",
  OrgDb = org.Rn.eg.db::org.Rn.eg.db,
  ont = "ALL",
  pAdjustMethod = "BH",
  pvalueCutoff = 1,
  qvalueCutoff = 1,  
  maxGSSize     = 500
)
hypo_ego_M4@result$term_size <- as.integer(sub("/.*", "", hypo_ego_M4@result$BgRatio))

DVC_ego_M25 <- enrichGO(
  gene = intersect(intersect(network_DVC[["M25"]], bg_dvc), gwas_dvc),
  universe = rownames(DVC_norm_counts),
  keyType = "SYMBOL",
  OrgDb = org.Rn.eg.db::org.Rn.eg.db,
  ont = "ALL",
  pAdjustMethod = "BH",
  pvalueCutoff = 1,
  qvalueCutoff = 1,
  maxGSSize = 500
)
DVC_ego_M25@result$term_size <- as.integer(sub("/.*", "", DVC_ego_M25@result$BgRatio))

In [ ]:
write.xlsx(hypo_ego_M4@result, "M4_overlap_GO.xlsx")
write.xlsx(DVC_ego_M25@result, "M25_overlap_GO.xlsx")

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)

# Data (long)
GO_DVC_hypo <- bind_rows(
  DVC_ego_M25@result %>%
    as_tibble() %>%
    arrange(p.adjust) %>%
    slice_head(n = 10) %>%
    transmute(Description, region = "DVC", p.adjust, GeneRatio),

  hypo_ego_M4@result %>%
    as_tibble() %>%
    arrange(p.adjust) %>%
    slice_head(n = 10) %>%
    transmute(Description, region = "Hypothalamus", p.adjust, GeneRatio)
) %>%
  group_by(Description, region) %>%
  summarise(
    p.adjust = min(p.adjust, na.rm = TRUE),
    GeneRatio = GeneRatio[which.min(p.adjust)],
    .groups = "drop"
  ) %>%
  mutate(
    GeneRatio_num = as.numeric(sub("/.*", "", GeneRatio)) / as.numeric(sub(".*/", "", GeneRatio)),
    p.adjust = pmax(p.adjust, .Machine$double.xmin)
  )

# Sorting: DVC-only first, Both middle, Hypo-only last
desc_order <- GO_DVC_hypo %>%
  group_by(Description) %>%
  summarise(
    in_DVC  = any(region == "DVC"),
    in_Hypo = any(region == "Hypothalamus"),
    best_p  = min(p.adjust, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(group = case_when(
    in_DVC & !in_Hypo ~ 1L,
    in_DVC &  in_Hypo ~ 2L,
    !in_DVC & in_Hypo ~ 3L
  )) %>%
  arrange(group, best_p) %>%
  pull(Description)

GO_DVC_hypo <- GO_DVC_hypo %>%
  mutate(
    Description = factor(Description, levels = rev(desc_order)),
    region = factor(region, levels = c("DVC", "Hypothalamus"))
  )

# Color scale limits: min..max p.adjust in THIS list (so "high" = least significant shown)
p_hi <- max(GO_DVC_hypo$p.adjust, na.rm = TRUE)
p_lo <- min(GO_DVC_hypo$p.adjust, na.rm = TRUE)

# Dotplot (match your example sizing/opacity + remove gridlines)
ggplot(GO_DVC_hypo, aes(x = region, y = Description)) +
  geom_point(aes(size = GeneRatio_num, color = p.adjust), alpha = 0.7) +
  scale_size_continuous(
    name = "GeneRatio",
    range = c(1, 4),
    breaks = c(0.01, 0.05, 0.10, 0.15),
    limits = c(0, max(0.15, max(GO_DVC_hypo$GeneRatio_num, na.rm = TRUE)))
  ) +
  scale_color_gradient(
    name   = "Adjusted\nP-value",
    low    = "#8f1d1e",
    high   = "#1e90ff",
    limits = c(p_lo, p_hi),
    oob    = squish,
    labels = label_scientific()
  ) +
  theme_classic(base_size = 6) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 6),
    axis.text.y = element_text(size = 6),
    legend.text = element_text(size = 6),
    legend.title = element_text(size = 6),
    legend.key.size = unit(2, "mm"),
    axis.line = element_line(color = "black", linewidth = 0.1),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1.5)
  ) +
  labs(x = NULL, y = NULL)

ggsave("GO_plot_adjusted.pdf", width = 8, height = 10, units = "cm", dpi = 600) 

# Module network

In [ ]:
library(igraph)
library(scales)

## ------------------------------------------------------------------
## parameters
## ------------------------------------------------------------------

mod_hypo <- "M4"
mod_dvc  <- "M25"
n_top    <- 10          # top kME genes per module

kME_col_hypo <- paste0("kME", mod_hypo)  # e.g. "kMEM4"
kME_col_dvc  <- paste0("kME", mod_dvc)   # e.g. "kMEM25"

## ------------------------------------------------------------------
## rare-variant BMI genes (for frame color)
## ------------------------------------------------------------------

rare_variant_bmi_genes <- c(
  "Mc4r","Mkks","Dido1","Bbs9","Zfhx3","Ttc8","Zfr2","Pde3b","Ano4","Ptprg",
  "Wdpcp","Slc12a5","Rab21","Kiaa0586","Bbs10","Trim32","Rab23","Ofd1","Calcr",
  "Tbx3","Sim1","Bbs7","Mks1","Gnas","Phf6","Gpc3","Wnt10b","Ubr2","Bbs12",
  "Ache","Cep290","Ntrk2","Hip1r","Bbs1","Entpd6","Gpr151","Bsn","Sdccag8",
  "Bltp1","Gipr","Apba1","Cep19","Sltm","Bbs2","Lepr","Bbs5","Prkag1","Sparc",
  "Zfp169","Lep","Pomc","Alms1","Robo1","Sh2b1","Vps13b","Dpp9","Ksr2","Gpr75",
  "Rapgef3","Prkar1a","Lztfl1","Arl6","Zbtb7b","Bbs4","Pcsk1","Grm7","Rai1",
  "Bdnf","Ankrd27","Uhmk1"
)

## ------------------------------------------------------------------
## 1) top kME genes for each module
## ------------------------------------------------------------------

genes_hypo <- network_hypo[[mod_hypo]]
genes_dvc  <- network_DVC[[mod_dvc]]

kME_hypo_mod <- kME_hypo[genes_hypo, kME_col_hypo, drop = FALSE]
kME_dvc_mod  <- kME_DVC[genes_dvc,  kME_col_dvc,  drop = FALSE]

kME_hypo_mod <- kME_hypo_mod[!is.na(kME_hypo_mod[, 1]), , drop = FALSE]
kME_dvc_mod  <- kME_dvc_mod[!is.na(kME_dvc_mod[, 1]),  , drop = FALSE]

top_hypo <- rownames(kME_hypo_mod)[order(kME_hypo_mod[, 1], decreasing = TRUE)]
top_hypo <- head(top_hypo, n_top)

top_dvc  <- rownames(kME_dvc_mod)[order(kME_dvc_mod[, 1], decreasing = TRUE)]
top_dvc  <- head(top_dvc, n_top)

all_genes <- union(top_hypo, top_dvc)

## ------------------------------------------------------------------
## 2) combined adjacency (mean of hypo + DVC) for selected genes
## ------------------------------------------------------------------

common_genes <- Reduce(
  intersect,
  list(all_genes, rownames(adjacency_hypo), rownames(adjacency_DVC))
)
all_genes <- common_genes

if (length(all_genes) < 2L) {
  stop("Not enough common genes between adjacency matrices and selected hubs.")
}

adj_h <- adjacency_hypo[all_genes, all_genes]
adj_d <- adjacency_DVC[all_genes, all_genes]

adj_h[is.na(adj_h)] <- 0
adj_d[is.na(adj_d)] <- 0

adj_comb <- (adj_h + adj_d) / 2
diag(adj_comb) <- 0

adj_vals <- adj_comb[adj_comb > 0]

if (length(adj_vals) == 0L) {
  stop("No positive edges among selected genes.")
}

thr <- quantile(adj_vals, 0.80)  # tweak 0.85 / 0.95 as desired

adj_thr <- adj_comb
adj_thr[adj_thr < thr] <- 0

idx <- which(adj_thr > 0, arr.ind = TRUE)

edges <- data.frame(
  from   = rownames(adj_thr)[idx[, "row"]],
  to     = colnames(adj_thr)[idx[, "col"]],
  weight = adj_thr[idx],
  stringsAsFactors = FALSE
)

edges <- edges[edges$from < edges$to, ]

## ------------------------------------------------------------------
## 3) vertex metadata: module, color, size, frame color
## ------------------------------------------------------------------

vertex_genes <- sort(unique(c(edges$from, edges$to)))

module <- ifelse(
  vertex_genes %in% top_hypo & vertex_genes %in% top_dvc, "Both",
  ifelse(
    vertex_genes %in% top_hypo, paste0("Hypo_", mod_hypo),
    paste0("DVC_", mod_dvc)
  )
)

module_levels <- c(paste0("Hypo_", mod_hypo),
                   paste0("DVC_", mod_dvc),
                   "Both")

module_cols <- setNames(
  c("dodgerblue2", "tomato", "purple"),
  module_levels
)

v_color <- unname(module_cols[module])

is_rare     <- vertex_genes %in% rare_variant_bmi_genes
v_frame_col <- ifelse(is_rare, "black", "grey40")

kME_hypo_vec <- rep(NA_real_, length(vertex_genes))
names(kME_hypo_vec) <- vertex_genes
kME_dvc_vec <- kME_hypo_vec

common_h <- intersect(vertex_genes, rownames(kME_hypo))
common_d <- intersect(vertex_genes, rownames(kME_DVC))

if (length(common_h) > 0) {
  kME_hypo_vec[common_h] <- kME_hypo[common_h, kME_col_hypo]
}
if (length(common_d) > 0) {
  kME_dvc_vec[common_d] <- kME_DVC[common_d, kME_col_dvc]
}

kME_comb <- pmax(abs(kME_hypo_vec), abs(kME_dvc_vec), na.rm = TRUE)

if (all(!is.finite(kME_comb))) {
  v_size <- rep(10, length(vertex_genes))
} else {
  kME_comb[!is.finite(kME_comb)] <- min(kME_comb[is.finite(kME_comb)])
  v_size <- rescale(kME_comb, to = c(8, 18))
}

vertices <- data.frame(
  name        = vertex_genes,
  module      = module,
  color       = v_color,
  size        = v_size,
  label       = vertex_genes,
  frame.color = v_frame_col,
  stringsAsFactors = FALSE
)

In [ ]:
## ------------------------------------------------------------------
## 4) build graph
## ------------------------------------------------------------------

g <- graph_from_data_frame(edges, directed = FALSE, vertices = vertices)

set.seed(196)
coords <- layout_with_fr(g)

## ---- PDF export ----
pdf("combined_hub_network_hypoM4_DVCM25.pdf", width = 7, height = 7)
par(mar = c(0, 0, 0, 0))  # optional: tighten margins

plot(
  g,
  layout              = coords,
  vertex.color        = V(g)$color,
  vertex.size         = V(g)$size,
  vertex.label        = V(g)$label,
  vertex.label.cex    = 0.7,
  vertex.label.font   = 2,
  vertex.label.family = "sans",
  vertex.frame.color  = V(g)$frame.color,
  edge.width          = 0.5 + 2 * E(g)$weight,
  edge.color          = rgb(0.2, 0.2, 0.2, alpha = 0.25),
  asp                 = 0,
  main                = paste0("Combined hub network: Hypothalamus-",
                               mod_hypo, " + DVC-", mod_dvc)
)

legend(
  "topleft",
  legend = c(paste0("Hypo ", mod_hypo),
             paste0("DVC ", mod_dvc),
             "Both",
             "Rare-variant BMI"),
  col    = c(module_cols[c(paste0("Hypo_", mod_hypo),
                           paste0("DVC_", mod_dvc),
                           "Both")],
             "black"),
  pch    = c(19, 19, 19, 1),
  pt.cex = 1.5,
  bty    = "n"
)

dev.off()

# Misc. plots

In [ ]:
options(repr.matrix.max.cols = 100)
DVC_module_treat_sign <- rbind(DVC_vehicle_cagri[2,], DVC_vehicle_reta[2,], DVC_vehicle_combo[2,])
hypo_module_treat_sign <- rbind(hypo_vehicle_cagri[2,], hypo_vehicle_reta[2,], hypo_vehicle_combo[2,])

In [ ]:
DVC_module_treat_sign <- DVC_module_treat_sign %>% select(where(~ any(. <= 0.05, na.rm = TRUE)))
hypo_module_treat_sign <- hypo_module_treat_sign %>% select(where(~ any(. <= 0.05, na.rm = TRUE)))

In [ ]:
DVC_module_treat_sign
hypo_module_treat_sign

In [ ]:
pval_heatmap <- function(df, cutoff = 0.05, title = NULL) {

  long <- df %>%
    as.data.frame() %>%
    tibble::rownames_to_column("metric") %>%
    tidyr::pivot_longer(-metric, names_to = "module", values_to = "p") %>%
    dplyr::mutate(
      p = as.numeric(p),
      p_plot = ifelse(p <= cutoff, p, NA_real_)
    )

  # Proper numeric ordering for M#
  module_levels <- long %>%
    dplyr::distinct(module) %>%
    dplyr::mutate(module_num = as.integer(sub("^M", "", module))) %>%
    dplyr::arrange(module_num) %>%
    dplyr::pull(module)

  long <- long %>%
    dplyr::mutate(
      module = factor(module, levels = module_levels),
      metric = factor(metric, levels = rev(unique(metric)))
    )

  ggplot(long, aes(x = module, y = metric, fill = p_plot)) +
    geom_tile(color = "grey90", linewidth = 0.3) +
    scale_fill_gradient(
      low = "#8f1d1e",   # smallest p
      high = "#1e90ff",  # near cutoff (0.05)
      limits = c(0, cutoff),
      na.value = "white",
      name = "p (≤ 0.05)"
    ) +
    labs(title = title, x = NULL, y = NULL) +
    theme_minimal(base_size = 10) +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      panel.grid = element_blank()
    )
}

pval_heatmap(DVC_module_treat_sign, cutoff = 0.05, title = "Table 1")
ggsave("DVC_module_treatment_sign.pdf", height = 5, width = 10)
pval_heatmap(hypo_module_treat_sign, cutoff = 0.05, title = "Table 1")
ggsave("hypo_module_treatment_sign.pdf", height = 5, width = 10)

# Tables

## Summaries

In [ ]:
summary_DVC <- stack(network_DVC)
colnames(summary_DVC) <- c("rat_gene", "module")
bmi_gwas_subset <- bmi_gwas[bmi_gwas$rat_gene %in% df$rat_gene,] %>% select(P, adj, rat_gene)
summary_DVC <- summary_DVC %>% left_join(bmi_gwas_subset, by = "rat_gene")
summary_DVC$rare_variant_bmi <- ifelse(summary_DVC$rat_gene %in% rare_variant_bmi_genes, TRUE, FALSE)
colnames(summary_DVC) <- c("gene", "module", "BMI_GWAS_pval", "BMI_GWAS_adjPval", "rare_variant_bmi_gene")

In [ ]:
summary_hypo <- stack(network_hypo)
colnames(summary_hypo) <- c("rat_gene", "module")
bmi_gwas_subset <- bmi_gwas[bmi_gwas$rat_gene %in% summary_hypo$rat_gene,] %>% select(P, adj, rat_gene)
summary_hypo <- summary_hypo %>% left_join(bmi_gwas_subset, by = "rat_gene")
summary_hypo$rare_variant_bmi <- ifelse(summary_hypo$rat_gene %in% rare_variant_bmi_genes, TRUE, FALSE)
colnames(summary_hypo) <- c("gene", "module", "BMI_GWAS_pval", "BMI_GWAS_adjPval", "rare_variant_bmi_gene")

In [ ]:
library(openxlsx)

### 1) Summaries: summary_DVC + summary_hypo ----
write.xlsx(
  list(
    DVC  = summary_DVC,
    hypo = summary_hypo
  ),
  file = "WGCNA_summaries.xlsx",
  overwrite = TRUE
)


## Treatment significance

In [ ]:
treat_DVC <- rbind(DVC_vehicle_cagri, DVC_vehicle_reta, DVC_vehicle_combo)
rownames(treat_DVC) <- c("pval_vehVScagri", "adjPval_vehVScagri", "pval_vehVSreta", "adjPval_vehVSreta", "pval_vehVScombo", "adjPval_vehVScombo")
treat_DVC <- as.data.frame(treat_DVC) %>% rownames_to_column("contrast")

In [ ]:
treat_hypo <- rbind(hypo_vehicle_cagri, hypo_vehicle_reta, hypo_vehicle_combo)
rownames(treat_hypo) <- c("pval_vehVScagri", "adjPval_vehVScagri", "pval_vehVSreta", "adjPval_vehVSreta", "pval_vehVScombo", "adjPval_vehVScombo")
treat_hypo <- as.data.frame(treat_hypo) %>% rownames_to_column("contrast")

In [ ]:
write.xlsx(
  list(
    DVC  = treat_DVC,
    hypo = treat_hypo
  ),
  file = "WGCNA_treatment_significance.xlsx",
  overwrite = TRUE
)

## BMI enrichment

In [ ]:
bmi_enrich_DVC <- enrichment_results_DVC
colnames(bmi_enrich_DVC) <- c("module", "overlap", "module_size", "pval", "adjPval")

In [ ]:
bmi_enrich_hypo <- enrichment_results_hypo
colnames(bmi_enrich_hypo) <- c("module", "overlap", "module_size", "pval", "adjPval")

In [ ]:
write.xlsx(
  list(
    DVC  = bmi_enrich_DVC,
    hypo = bmi_enrich_hypo
  ),
  file = "WGCNA_BMI_enrichment.xlsx",
  overwrite = TRUE
)

## kME

In [ ]:
kME_DVC_export <- as.data.frame(kME_DVC) %>% rownames_to_column("gene")

In [ ]:
kME_hypo_export <- as.data.frame(kME_hypo) %>% rownames_to_column("gene")

In [ ]:
write.xlsx(
  list(
    DVC  = kME_DVC_export,
    hypo = kME_hypo_export
  ),
  file = "WGCNA_kME.xlsx",
  overwrite = TRUE
)